# Evals deterministas — recorrido paso a paso

Corre las celdas **una por una** (Shift+Enter) y mira el resultado antes de seguir con la siguiente. Nada de esto llama a internet ni necesita API key — todo lee JSON y el `.parquet` local.

Este notebook asume que vive en `agentic-evaluation-workshop/notebooks/` (junto a `data/` y `eval/`, un nivel arriba).

In [ ]:
# Setup -- corre esta celda primero, una sola vez
import sys, os, json

sys.path.insert(0, os.path.abspath("../eval"))
print("listo, ya puedes importar los módulos de eval/")

## Paso 1 — cargar un golden y un extracted (Encuentro 1: Rivendell)

In [ ]:
with open("../data/golden/golden_encounter_riv001.json") as f:
    golden_1 = json.load(f)

with open("../data/golden/extracted_encounter_1_good.json") as f:
    extracted_1_good = json.load(f)

with open("../data/golden/extracted_encounter_1_bad.json") as f:
    extracted_1_bad = json.load(f)

golden_1["encounter_id"], extracted_1_good["encounter_id"]

Deberías ver `('RIV-001', 'RIV-001')`. Solo confirmamos que los archivos cargaron bien, todavía no evaluamos nada.

## Paso 2 — eval de diagnósticos

In [ ]:
from eval_diagnoses import eval_diagnoses, load_icd10_catalog_parquet

catalog = load_icd10_catalog_parquet("../data/Diagnosis.parquet")
eval_diagnoses(golden_1, extracted_1_good, catalog)

Con el mock "good" esperamos `precision=1.0, recall=1.0` y las listas vacías. Ahora el mock "bad" (el que alucina "transformación en espectro"):

In [ ]:
eval_diagnoses(golden_1, extracted_1_bad, catalog)

Deberías ver `invalid_codes` con un código inventado, y `false_negative_codes` con algo que el mock malo se saltó.

## Paso 3 — eval de vitals

In [ ]:
from eval_vitals import eval_vitals

eval_vitals(golden_1, extracted_1_good)   # esperado: todo en 1.0, listas vacías

In [ ]:
eval_vitals(golden_1, extracted_1_bad)    # esperado: value_mismatch en heart_rate_bpm

## Paso 4 — calibración de confidence

In [ ]:
from eval_confidence import eval_confidence_calibration

eval_confidence_calibration(extracted_1_good)   # esperado: []

In [ ]:
eval_confidence_calibration(extracted_1_bad)    # esperado: un flag de confidence "high" con evidencia de informante no clínico

## Paso 5 — trayectoria (tool calls)

In [ ]:
from eval_trajectory import eval_trajectory

with open("../data/golden/trace_encounter_1_good.json") as f:
    trace_1_good = json.load(f)

with open("../data/golden/trace_encounter_1_bad.json") as f:
    trace_1_bad = json.load(f)

eval_trajectory(golden_1, trace_1_good)   # esperado: sin mismatches

In [ ]:
eval_trajectory(golden_1, trace_1_bad)    # esperado: count_mismatches + argument_mismatches

---
## Ahora lo mismo con el Encuentro 2 (Casas de Curación)

Este encuentro tiene una trampa distinta: no es alucinación, es **diagnóstico prematuro** (el mock malo le pone código a TEPT antes de que se cumpla el criterio temporal).

In [ ]:
with open("../data/golden/golden_encounter_hou002.json") as f:
    golden_2 = json.load(f)

with open("../data/golden/extracted_encounter_2_good.json") as f:
    extracted_2_good = json.load(f)

with open("../data/golden/extracted_encounter_2_bad.json") as f:
    extracted_2_bad = json.load(f)

golden_2["encounter_id"], extracted_2_good["encounter_id"]

### Diagnósticos

In [ ]:
eval_diagnoses(golden_2, extracted_2_good, catalog)

In [ ]:
eval_diagnoses(golden_2, extracted_2_bad, catalog)
# esperado: premature_diagnosis_violations con "Trastorno de estrés postraumático" / F43.10

### Vitals

In [ ]:
eval_vitals(golden_2, extracted_2_good)

In [ ]:
eval_vitals(golden_2, extracted_2_bad)

### Confidence

In [ ]:
eval_confidence_calibration(extracted_2_good)

In [ ]:
eval_confidence_calibration(extracted_2_bad)

### Trayectoria

In [ ]:
with open("../data/golden/trace_encounter_2_good.json") as f:
    trace_2_good = json.load(f)

with open("../data/golden/trace_encounter_2_bad.json") as f:
    trace_2_bad = json.load(f)

eval_trajectory(golden_2, trace_2_good)

In [ ]:
eval_trajectory(golden_2, trace_2_bad)
# esperado: count_mismatch en get_patient_history (0 en vez de 1)

---
## Lo que NO está en este notebook

El chequeo de fidelidad (`eval/requires_internet/eval_hpi_judge.py`) necesita llamar a la API de Claude, así que lo dejamos aparte a propósito. Cuando decidan usarlo, la idea es correrlo *una vez* con internet antes del taller y guardar los resultados en disco, no llamarlo en vivo durante la presentación.